# Обучение VAE / GAN / DDPM на Kaggle (2x T4)

Ноутбук для актуальной версии репозитория — все фиксы уже в коде
(AMP, DDP, BCEWithLogitsLoss и т.д.), патч-ячейка не нужна.

Порядок: проверка окружения -> dry run -> VAE -> GAN -> DDPM.

In [ ]:
import os
import subprocess

if not os.path.isdir("ImageModel"):
    subprocess.run(
        ["git", "clone", "https://github.com/Magmucot/ImageModel.git"],
        check=True,
    )
os.chdir("ImageModel")
!ls

In [ ]:
!pip install -q pyyaml pillow matplotlib

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "x", torch.cuda.device_count())
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ), "GB")

## Поиск датасета

Запустите ячейку ниже и сверьте реальный путь с переменной `DATA`
в следующей ячейке (Kaggle монтирует датасеты как `/kaggle/input/<slug>/...`).

In [ ]:
!find /kaggle/input -maxdepth 3 -type d | head -30

In [ ]:
# Проверяем путь до запуска обучения
from pathlib import Path

DATA = "/kaggle/input/ffhq-face-data-set/thumbnails128x128"

if not Path(DATA).is_dir():
    raise FileNotFoundError(
        f"Датасет не найден: {DATA}\n"
        "Смотрите вывод предыдущей ячейки и поправьте путь."
    )

n_images = sum(
    1 for p in Path(DATA).rglob("*")
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
print(f"Найдено изображений: {n_images}")
assert n_images > 0, "В папке нет изображений"

## Запуск обучения

1. `--dry_run` DDPM — быстрая проверка пайплайна без датасета.
2. VAE -> GAN -> DDPM на двух T4 через `torchrun`.

In [ ]:
%env OMP_NUM_THREADS=1

!python DDPM/train.py --dry_run   # работает без датасета

In [ ]:
print("VAE---------------------------")
!torchrun --standalone --nproc_per_node=2 VAE/train.py  --data_root "$DATA" --config configs/vae.yaml

In [ ]:
print("GAN---------------------------")
!torchrun --standalone --nproc_per_node=2 GAN/train.py  --data_root "$DATA" --config configs/gan.yaml

In [ ]:
print("DDPM--------------------------")
!torchrun --standalone --nproc_per_node=2 DDPM/train.py --data_root "$DATA" --config configs/ddpm.yaml